# Live URL detection
This calls the production orchestrator. A blocked page should return `crawl_failed`; it must not be interpreted as phishing.

In [ ]:
%pip install -r ../requirements-browser.txt
# Run once if Chromium is absent:
# !python -m playwright install chromium

In [ ]:
from pathlib import Path
import sys, asyncio, json
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from persianphish_detector.config import DetectorConfig
from persianphish_detector.orchestrator import Detector

In [ ]:
async def detect_url(url):
    detector = Detector(DetectorConfig.from_env(ROOT))
    try:
        return (await detector.detect(url)).to_dict()
    finally:
        await detector.close()
result = await detect_url('https://soft98.ir')
print(json.dumps(result, ensure_ascii=False, indent=2))

For service testing, start `python -m persianphish_detector serve --port 8088`, then use the next cell.

In [ ]:
import httpx
response = httpx.post('http://127.0.0.1:8088/v1/detect', json={'url':'https://example.com'}, timeout=40)
response.raise_for_status()
response.json()